### MediaPipe Extractor

In [27]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp 

In [28]:
mp_holist = mp.solutions.holistic 
mp_draw = mp.solutions.drawing_utils

In [29]:
def mediapipe_detection(img, model):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img.flags.writeable = False                 
    result = model.process(img)                 # Make prediction
    img.flags.writeable = True                   
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
    return img, result

In [30]:
def draw_landmarks(img, result):
    mp_draw.draw_landmarks(img, result.face_landmarks, mp_holist.FACEMESH_CONTOURS) # Draw face connections
    mp_draw.draw_landmarks(img, result.pose_landmarks, mp_holist.POSE_CONNECTIONS) # Draw pose connections
    mp_draw.draw_landmarks(img, result.left_hand_landmarks, mp_holist.HAND_CONNECTIONS) # Draw left hand connections
    mp_draw.draw_landmarks(img, result.right_hand_landmarks, mp_holist.HAND_CONNECTIONS) # Draw right hand connections

In [31]:
def draw_styled_landmarks(img, result):
    mp_draw.draw_landmarks(img, result.face_landmarks, mp_holist.FACEMESH_CONTOURS, 
                             mp_draw.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1), # color the joint 
                             mp_draw.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1) #color the connection
                             ) 
    # mp_draw.draw_landmarks(img, result.face_landmarks, mp_holist.FACEMESH_CONTOURS, 
    #                          mp_draw.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1), # color the joint 
    #                          mp_draw.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1) #color the connection
    #                          ) 
    
    mp_draw.draw_landmarks(img, result.pose_landmarks, mp_holist.POSE_CONNECTIONS,
                             mp_draw.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_draw.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 
    mp_draw.draw_landmarks(img, result.left_hand_landmarks, mp_holist.HAND_CONNECTIONS, 
                             mp_draw.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_draw.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    mp_draw.draw_landmarks(img, result.right_hand_landmarks, mp_holist.HAND_CONNECTIONS, 
                             mp_draw.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_draw.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             ) 

In [32]:
mp_holist.POSE_CONNECTIONS

frozenset({(0, 1),
           (0, 4),
           (1, 2),
           (2, 3),
           (3, 7),
           (4, 5),
           (5, 6),
           (6, 8),
           (9, 10),
           (11, 12),
           (11, 13),
           (11, 23),
           (12, 14),
           (12, 24),
           (13, 15),
           (14, 16),
           (15, 17),
           (15, 19),
           (15, 21),
           (16, 18),
           (16, 20),
           (16, 22),
           (17, 19),
           (18, 20),
           (23, 24),
           (23, 25),
           (24, 26),
           (25, 27),
           (26, 28),
           (27, 29),
           (27, 31),
           (28, 30),
           (28, 32),
           (29, 31),
           (30, 32)})

In [33]:
cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holist.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        ret, frame = cap.read()

        image, results = mediapipe_detection(frame, holistic)
        print(results)
        
        draw_styled_landmarks(image, results)

        cv2.imshow('OpenCV Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.soluti

In [34]:
cap.release()
cv2.destroyAllWindows()

In [35]:
results.pose_landmarks.landmark[0].visibility

0.9795895218849182

In [36]:
len(results.pose_landmarks.landmark)

33

In [37]:
def extract_keypoints(results):
    pose=np.array([[res.x,res.y] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*2)
    left_hnd=np.array([[res.x,res.y] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*2)
    right_hnd=np.array([[res.x,res.y] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*2)
    face=np.array([[res.x,res.y] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*2)
    return np.concatenate([pose,left_hnd,right_hnd,face])
# concatenating for the model to detect the sign language

In [38]:
extract_keypoints(results).shape

(1086,)

In [39]:
import os 
video_dir = "C:/Users/araya/Desktop/keypoints/video_extract"
video_list = []
video_list = os.listdir(video_dir)

len(video_list)

40

In [40]:
video_list

['กลัว.mp4',
 'กิน.mp4',
 'ขยัน.mp4',
 'ขอบคุณ.mp4',
 'ขอโทษ.mp4',
 'ขี้เกียจ.mp4',
 'ขโมย.mp4',
 'คิดถึง.mp4',
 'คุณ พรุ่งนี้อย่าลืมส่งการบ้านคณิตศาสตร์ให้ครู.mp4',
 'คุณ.mp4',
 'คุณช่วยแปลภาษาอังกฤษเป็นภาษาไทยให้ฉันด้วย.mp4',
 'ง่าย.mp4',
 'ฉลาด.mp4',
 'ฉัน.mp4',
 'ฉันตื่นนอน 6 โมงเช้าทุกวัน.mp4',
 'ฉันฝึกเขียนภาษาไทยจนเขียนได้ดีขึ้น.mp4',
 'ชอบ.mp4',
 'ดีใจ.mp4',
 'บ้าน.mp4',
 'พรุ่งนี้.mp4',
 'ภาษาอังกฤษเป็นเรื่องยากสำหรับคนหูหนวก.mp4',
 'ยาก.mp4',
 'ยินดีด้วย.mp4',
 'รถยนต์.mp4',
 'รัก.mp4',
 'ลาก่อน.mp4',
 'วิชาคณิตศาสตร์เป็นวิชาที่เรียนยากมาก.mp4',
 'สบายดี.mp4',
 'สวย.mp4',
 'สวัสดี.mp4',
 'สอบ.mp4',
 'หล่อ.mp4',
 'อยาก.mp4',
 'เก่ง.mp4',
 'เที่ยว.mp4',
 'โรงเรียน.mp4',
 'ใช่.mp4',
 'ไป.mp4',
 'ไม่อยาก.mp4',
 'ไม่ใช่.mp4']

In [41]:
# Path for exported data, numpy arrays
Model_Data=os.path.join('Data for different actions')

actions = np.array(video_list)

no_of_seqs = 1

# 30 frames in length
# seq_length = 210
seq_length = 250

In [42]:
actions

array(['กลัว.mp4', 'กิน.mp4', 'ขยัน.mp4', 'ขอบคุณ.mp4', 'ขอโทษ.mp4',
       'ขี้เกียจ.mp4', 'ขโมย.mp4', 'คิดถึง.mp4',
       'คุณ พรุ่งนี้อย่าลืมส่งการบ้านคณิตศาสตร์ให้ครู.mp4', 'คุณ.mp4',
       'คุณช่วยแปลภาษาอังกฤษเป็นภาษาไทยให้ฉันด้วย.mp4', 'ง่าย.mp4',
       'ฉลาด.mp4', 'ฉัน.mp4', 'ฉันตื่นนอน 6 โมงเช้าทุกวัน.mp4',
       'ฉันฝึกเขียนภาษาไทยจนเขียนได้ดีขึ้น.mp4', 'ชอบ.mp4', 'ดีใจ.mp4',
       'บ้าน.mp4', 'พรุ่งนี้.mp4',
       'ภาษาอังกฤษเป็นเรื่องยากสำหรับคนหูหนวก.mp4', 'ยาก.mp4',
       'ยินดีด้วย.mp4', 'รถยนต์.mp4', 'รัก.mp4', 'ลาก่อน.mp4',
       'วิชาคณิตศาสตร์เป็นวิชาที่เรียนยากมาก.mp4', 'สบายดี.mp4',
       'สวย.mp4', 'สวัสดี.mp4', 'สอบ.mp4', 'หล่อ.mp4', 'อยาก.mp4',
       'เก่ง.mp4', 'เที่ยว.mp4', 'โรงเรียน.mp4', 'ใช่.mp4', 'ไป.mp4',
       'ไม่อยาก.mp4', 'ไม่ใช่.mp4'], dtype='<U49')

In [43]:
# just creating the folders and sub folders

for action in actions: 
    try: 
        os.makedirs(os.path.join(Model_Data, action))
    except:
        pass

In [44]:
# import os
# augment_dir = "C:/Users/araya/Desktop/augments"

# augment_list = []
# augment_list = os.listdir(augment_dir)
# augment_list

In [45]:
# actions = list(actions)

In [46]:
# for x in augment_list:
#     # print(x)
#     actions.append(x)
# actions

In [47]:
# actions = np.array(actions)
# actions

Collecting keypoint values for Training nd Testing

In [48]:
# Define the directory where your videos are stored
directory = "C:/Users/araya/Desktop/keypoints/video_extract"

In [49]:
directory

'C:/Users/araya/Desktop/keypoints/video_extract'

In [50]:
for filename in actions:
    print(directory + '/' + filename)

C:/Users/araya/Desktop/keypoints/video_extract/กลัว.mp4
C:/Users/araya/Desktop/keypoints/video_extract/กิน.mp4
C:/Users/araya/Desktop/keypoints/video_extract/ขยัน.mp4
C:/Users/araya/Desktop/keypoints/video_extract/ขอบคุณ.mp4
C:/Users/araya/Desktop/keypoints/video_extract/ขอโทษ.mp4
C:/Users/araya/Desktop/keypoints/video_extract/ขี้เกียจ.mp4
C:/Users/araya/Desktop/keypoints/video_extract/ขโมย.mp4
C:/Users/araya/Desktop/keypoints/video_extract/คิดถึง.mp4
C:/Users/araya/Desktop/keypoints/video_extract/คุณ พรุ่งนี้อย่าลืมส่งการบ้านคณิตศาสตร์ให้ครู.mp4
C:/Users/araya/Desktop/keypoints/video_extract/คุณ.mp4
C:/Users/araya/Desktop/keypoints/video_extract/คุณช่วยแปลภาษาอังกฤษเป็นภาษาไทยให้ฉันด้วย.mp4
C:/Users/araya/Desktop/keypoints/video_extract/ง่าย.mp4
C:/Users/araya/Desktop/keypoints/video_extract/ฉลาด.mp4
C:/Users/araya/Desktop/keypoints/video_extract/ฉัน.mp4
C:/Users/araya/Desktop/keypoints/video_extract/ฉันตื่นนอน 6 โมงเช้าทุกวัน.mp4
C:/Users/araya/Desktop/keypoints/video_extract/ฉันฝึกเ

In [51]:
# # Set mediapipe model 
# for action in actions:
#     video_path = os.path.join("C:/Users/araya/Desktop/keypoints/video_extract", action)
#     cap = cv2.VideoCapture(video_path)
#     cap.set(cv2.CAP_PROP_FPS, 60)
#     length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     print("LENGTH:" + str(length))
#     # keypoints = []

#     if not cap.isOpened():
#         print(f"Error opening video file: {video_path}")
#         continue

#     with mp_holist.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
#         for seq in range(no_of_seqs):
#             for frame_num in range(seq_length):

#                 ret, frame = cap.read()
#                 if not ret:
#                     print(f"End of video {video_path}")
#                     break
                
#                 img, results = mediapipe_detection(frame, holistic)
#                 draw_styled_landmarks(img, results)

#                 # print(frame_num)

#                 if frame_num == 0: 
#                     cv2.putText(img, 'DATA COLLECTION STARTED', (120,200), 
#                                 cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
#                     cv2.putText(img, f'Collecting frames for - {action} Sequence Number - {seq}', (15,12), 
#                                 cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
#                     cv2.imshow('OpenCV Window', img)
#                     cv2.waitKey(2000)  # 2 seconds delay for setup
#                 else: 
#                     cv2.putText(img, f'Collecting frames for - {action} Sequence Number - {seq}', (15,12), 
#                                 cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
#                     cv2.imshow('OpenCV Window', img)

#                 keypoints = extract_keypoints(results)
#                 # keypoints.append(results)
#                 npy_path = os.path.join(Model_Data, action, f"frame_{frame_num}.npy")
#                 os.makedirs(os.path.dirname(npy_path), exist_ok=True)
#                 np.save(npy_path, keypoints)

#                 if cv2.waitKey(1) & 0xFF == ord('q'):
#                     break

#             if not ret:
#                 break

#     cap.release()
#     cv2.destroyAllWindows()

In [52]:
# Set mediapipe model 
for action in actions:
    video_path = os.path.join("C:/Users/araya/Desktop/keypoints/video_extract", action)
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_FPS, 60)
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print("LENGTH:" + str(length))
    keypoints = []

    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        continue

    with mp_holist.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        for seq in range(no_of_seqs):
            for frame_num in range(seq_length):

                ret, frame = cap.read()
                if not ret:
                    print(f"End of video {video_path}")
                    break
                
                img, results = mediapipe_detection(frame, holistic)
                draw_styled_landmarks(img, results)

                # print(frame_num)

                if frame_num == 0: 
                    cv2.putText(img, 'DATA COLLECTION STARTED', (120,200), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
                    cv2.putText(img, f'Collecting frames for - {action} Sequence Number - {seq}', (15,12), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    cv2.imshow('OpenCV Window', img)
                    cv2.waitKey(2000)  # 2 seconds delay for setup
                else: 
                    cv2.putText(img, f'Collecting frames for - {action} Sequence Number - {seq}', (15,12), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    cv2.imshow('OpenCV Window', img)

                x = extract_keypoints(results)
                keypoints.append(x)
                npy_path = os.path.join(Model_Data, action, f"{action.split(".")[0]}.npy")
                os.makedirs(os.path.dirname(npy_path), exist_ok=True)
                np.save(npy_path, keypoints)

                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

            if not ret:
                break

    cap.release()
    cv2.destroyAllWindows()

LENGTH:79
End of video C:/Users/araya/Desktop/keypoints/video_extract\กลัว.mp4
LENGTH:105
End of video C:/Users/araya/Desktop/keypoints/video_extract\กิน.mp4
LENGTH:171
End of video C:/Users/araya/Desktop/keypoints/video_extract\ขยัน.mp4
LENGTH:217
End of video C:/Users/araya/Desktop/keypoints/video_extract\ขอบคุณ.mp4
LENGTH:209
End of video C:/Users/araya/Desktop/keypoints/video_extract\ขอโทษ.mp4
LENGTH:186
End of video C:/Users/araya/Desktop/keypoints/video_extract\ขี้เกียจ.mp4
LENGTH:111
End of video C:/Users/araya/Desktop/keypoints/video_extract\ขโมย.mp4
LENGTH:202
End of video C:/Users/araya/Desktop/keypoints/video_extract\คิดถึง.mp4
LENGTH:231
End of video C:/Users/araya/Desktop/keypoints/video_extract\คุณ พรุ่งนี้อย่าลืมส่งการบ้านคณิตศาสตร์ให้ครู.mp4
LENGTH:173
End of video C:/Users/araya/Desktop/keypoints/video_extract\คุณ.mp4
LENGTH:247
End of video C:/Users/araya/Desktop/keypoints/video_extract\คุณช่วยแปลภาษาอังกฤษเป็นภาษาไทยให้ฉันด้วย.mp4
LENGTH:194
End of video C:/Users/ara

In [ ]:
cap.release()
cv2.destroyAllWindows()